# RepairScope-X — Combined MDLM and BD3LM (Notebook 18+19, v6)
### Two real diffusion architectures, one Colab notebook, short restartable cells

This replaces the repeatedly failing separate 18/19 notebooks. **Do not run earlier notebooks in the same Colab runtime.**

- Uses frozen 24 problems (8 Countdown / 8 GSM8K / 8 SVAMP), verified by their original SHA-256.
- Loads **one 0.5B diffusion model at a time**, runs it, exports it, *unloads it*, then switches to the other model.
- Each execution batch has **at most four real GPU inference trials** with a soft eight-minute cutoff (checked after each trial). No cell runs an entire 1½-hour sweep.
- Resumes using Google Drive files, with one fsync'd JSONL record per completed trial. Restarting a runtime and repeating a batch skips completed trial IDs.
- Uses only the official pinned A2D Qwen2 model class, `MDLMSampler` / `BD3LMSampler`, and their official scheduler code. No custom decoding loops or Hugging Face `trust_remote_code`.
- Adds a checkpoint manifest for every model and records *real model.forward invocations*, wall time, preserved tokens, task validity, and full outputs.
- **MDLM**: native infilling with `tail25`, `tail50`, `middle25`, `global`.
- **BD3LM**: native prefix-conditioned suffix generation with `tail25`, `tail50`, `tail75`, `global`.

### Fresh runtime setup
In Colab: **Runtime → Disconnect and delete runtime**, then choose **T4 GPU** and open this notebook. Run each cell in order. Google Drive permission is necessary for durable checkpoints: the notebook will stop rather than silently saving only under `/content`.

**If a GPU cell errors:** Do not delete the Google Drive experiment folder. Rerun the setup/load cells after restarting Colab; executed trial IDs will be skipped. An error JSONL and diagnostics ZIP are available via the diagnostic cell near the end.

**Important:** The tests performed here before delivery are static validation and CPU simulated-inference tests, not a verified T4 run. Real checkpoint loading and real CUDA sampling must still be confirmed by the first two smoke-test cells in Colab.

## 0. Environment check and isolated dependencies

In [ ]:
# Must be first code cell; does not import NumPy into the notebook kernel.
import os, sys, subprocess, pathlib
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'  # public model snapshots
if sys.version_info < (3,10):
    raise RuntimeError('Python 3.10 or newer is required.')
check="import numpy, torch; assert numpy.asarray([1,2]).sum()==3; print('NumPy',numpy.__version__,'Torch',torch.__version__)"
r=subprocess.run([sys.executable,'-c',check],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-7000:])
    raise RuntimeError('Scientific Python environment already corrupted. Colab: Runtime > Disconnect and delete runtime; reconnect; do NOT reinstall NumPy inside a live runtime.')
print(r.stdout)
# Narrow, pinned Python interfaces only; never reinstall PyTorch, NumPy, SciPy or Pandas.
packages=['transformers==4.57.0','accelerate==1.11.0',
          'huggingface-hub==0.35.3','tokenizers==0.22.1','safetensors==0.6.2']
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','--no-deps',*packages],check=True)
preflight="import torch,transformers,accelerate,huggingface_hub,tokenizers,safetensors; from transformers import AutoTokenizer; from transformers.processing_utils import Unpack; from transformers.utils import TransformersKwargs; from transformers.modeling_attn_mask_utils import _prepare_4d_attention_mask; print('model interface',transformers.__version__)"
r=subprocess.run([sys.executable,'-c',preflight],capture_output=True,text=True)
if r.returncode:
    print(r.stderr[-9000:])
    raise RuntimeError('Model library import preflight failed. Start from a fresh runtime and report this error rather than running GPU stages.')
print(r.stdout)

## 1. Imports, official source, and persistent Google Drive

In [ ]:
import base64, zlib, json, re, math, ast as pyast, gc, time, random, traceback, hashlib, zipfile, csv, shutil, subprocess, sys, importlib.util, os
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import torch, transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
assert torch.cuda.is_available(), 'Choose T4 GPU and restart from the beginning.'
assert transformers.__version__=='4.57.0', 'Transformers version mismatch; restart with a fresh runtime.'
GPU=torch.cuda.get_device_name(0); VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
assert VRAM >= 7, f'Insufficient GPU memory: {VRAM:.2f} GB'
print('GPU',GPU,'VRAM',round(VRAM,2),'GB','torch',torch.__version__)
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive/RepairScopeX/RSX18_19_Combined_v6')
DRIVE.mkdir(parents=True,exist_ok=True)
# Verify persistent destination is writable; do not fall back to ephemeral /content.
(DRIVE/'drive_write_test.txt').write_text('ok',encoding='utf-8')
assert (DRIVE/'drive_write_test.txt').read_text()=='ok'
SOURCE_COMMIT='ca176752fbceec49c6b4777a2c18ae88e4eb10ed'
ROOT=Path('/content/rsx_v6_official_repo')
if not (ROOT/'.git').exists():
    ROOT.mkdir(parents=True,exist_ok=True)
    subprocess.run(['git','init','-q',str(ROOT)],check=True)
    subprocess.run(['git','-C',str(ROOT),'remote','add','origin','https://github.com/ZHZisZZ/dllm.git'],check=True)
try:
    sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],stderr=subprocess.DEVNULL,text=True).strip()
except subprocess.CalledProcessError:
    sha=''
if sha!=SOURCE_COMMIT:
    for retry in range(3):
        try:
            subprocess.run(['git','-C',str(ROOT),'fetch','--quiet','--depth','1','origin',SOURCE_COMMIT],check=True,timeout=240)
            subprocess.run(['git','-C',str(ROOT),'checkout','-q','--force','FETCH_HEAD'],check=True,timeout=80)
            break
        except Exception:
            if retry==2: raise
            time.sleep(2**retry)
sha=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
assert sha==SOURCE_COMMIT
print('Official code SHA',sha)

## 2. Official inference-only sampler imports (no full training package)

In [ ]:
# Exact official sampler files, copied BYTE-FOR-BYTE from the pinned upstream commit.
SHIM=Path('/content/rsx_v6_inference_only')
files=[
    'dllm/core/samplers/__init__.py','dllm/core/samplers/base.py',
    'dllm/core/samplers/mdlm.py','dllm/core/samplers/bd3lm.py',
    'dllm/core/samplers/utils.py','dllm/core/schedulers/__init__.py',
    'dllm/core/schedulers/alpha.py','dllm/core/schedulers/kappa.py',
]
for rel in files:
    source=ROOT/rel; destination=SHIM/rel
    assert source.exists(),source
    destination.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,destination)
    assert hashlib.sha256(source.read_bytes()).digest()==hashlib.sha256(destination.read_bytes()).digest()
for rel in ['dllm/__init__.py','dllm/core/__init__.py']:
    f=SHIM/rel;f.parent.mkdir(parents=True,exist_ok=True)
    f.write_text('# Inference namespace, without unrelated training imports.\n')
# Model class from exact official source, no checkpoint `trust_remote_code` needed.
MODEL_FILE=ROOT/'dllm/pipelines/a2d/models/qwen2/modeling_qwen2.py'
assert MODEL_FILE.is_file()
sys.path.insert(0,str(SHIM))
from dllm.core.samplers import MDLMSampler, MDLMSamplerConfig, BD3LMSampler, BD3LMSamplerConfig
model_module_name='rsx_v6_a2d_qwen2_source'
if model_module_name not in sys.modules:
    spec=importlib.util.spec_from_file_location(model_module_name,str(MODEL_FILE))
    module=importlib.util.module_from_spec(spec)
    sys.modules[model_module_name]=module
    spec.loader.exec_module(module)
else:
    module=sys.modules[model_module_name]
A2DQwen2Config=module.A2DQwen2Config
A2DQwen2LMHeadModel=module.A2DQwen2LMHeadModel
assert A2DQwen2Config.model_type=='a2d-qwen2'
print('Official samplers:',MDLMSampler.__name__,BD3LMSampler.__name__)

## 3. Frozen 24-problem corpus (same as earlier 18/19 runs)

In [ ]:
PAYLOAD_SHA="cc04b82051f62f8640b92c418571d5efd3fd89334bf4be126bfae6f508ff5328"
_encoded="eNq1WW2P27gR/iuEsUW+eBeWLPkNRYttkyAX3KXFbooULQofLdMWYUlUScqOcrj/3mdIyZad7a1tnIFssmtyhjPPPPO2+aVnVKUTYXqzf//SK7VaZCKfy2Vv1ktUVdil2hWzhGdyobmVqpgFk0E8iwa9fm/JLTfCdm/OV1qYFGcl11bSfTo9SNOJVnlJQv8wgtlUsKLKF0IbFk77DH94sWQTJr7yxGY1U0WCW4pBalklXsByvRaWBeEDexK20gXjbIs3loxradNcWJlAQQlTDN50GgW+djhkP7//4dPjj/PHT89f3j3N2B/963/6+QGmecW9GTQPfu2fj8bk9wcjiPosGHg0wuFZcNwMjcvAiMY3oEafDT0W8TlQDAe3wgKaLwIjvAE1kCPBuGHG4Bw44vBmeJDqiwAJbsCOYNTJlbNKxzC6GUGiy/CIb5IsYXQJGvHN0IgvRGMY/P5wDMGMsGFHcFa+RDdLl+jS8jG9AT2IH/El9eM+HN4KEFL9PSJrk082M6u5LGbxcHQMgjs8z/WfuLUs5QZWl4IXlWVlxgvr7sFBbpk0LB4M2ErAzUW9//6BsYCZ/1ZcC7ZSyjK1ajQYlvCC5XwjcJmtNc9N5xByP6yYtEBuIwzBe3qDIPbSp0dsUVkrtOcp26y/PzEiywzs0ewORE7VjuVVkrKlwlOIllO7gu8shVsHT//cLU9BGA9eQXw0GsfXIv6R5zBmUdWGjZi9N6nU1pscD/4Aj1ZA6HMqaiY4LE+UseyOuhP7sPcG5IIzJa+7do9eM3ocDMdXG63Sgi1cxIIQRiWKQM95wrVShel7Y3dCrlNZrBE5pCEG5gf2waVJAWOTDUhWMAHKZE2mUfysC0ujh8mCRfBvtRJaFIiqRibj3TVe0IIva4fTUmRyK3T9wL6QaihYaClWyNBM8C1MJJ0baRM6pXyVycZ9VqaqEH0X+0xam5GgwoFmz1ZsBZywXhhvS/AaXlp86xgIydbcg1WIypcmR1wxUBa+ORRsn3zxKPRbQS1yBIIA+g7BbiSj+LWEH4ejayP5meygT1ujeE4lk37K4WPN/so177OPvBDWV8CPQusaNQIAwc9oFs9GD5TDB49PVViibytwN0YlhTFJlXHri2amjHHveP30FOphlvn8lCuvIFH5QhZOQupWs6LMYUrzYi2ME6eUpzs5QyAmLoV8lKqakC61TES3nk5fze3o6tx+hkFauXpq5FdmJaU6fsh5UbMlr8AL6i90jJg7FGFpUmlie1Y7QZtqIdx53xcwkk1SmS1xaV/JCN0uZ8LgVc6Mr87+90g6gnTFcwkrtagK3OizvwuqudK0WAcjVoIJS5cwCwTsHjkA05ON8I0v5dnKd5WGMjhPeb6o4ISmbPog3hi2VqSsCXWIyCMVD3pT9JqlWhNQ/CVpnhl1rMIgtzVoauTSFQfesM0lamtTNxHaJ9qiSxE4GIBut/QKPMuOmser7AriyfVF2Igybbq1Fist4Te3SiNP2Y5TLFLh/iFYEZZUVUb4JKbKm4nEIhuY2oqmt6MImYo4Wqod9VXbqb3QTAX8829kOe43NqEX+R7mq6fQ65rh7SVNDL7yHaz11ZJI7vOjg73vagdN3qyuoq6b+yp07Bmutw7tlMaUdUczi6TBDIbbtM/sDjWB7f6f/V729DWn97SSHSHzqvUYM9d8CaKYo18ewLzXkjeKwquTF7UykeZNm71En5BVmyoDbHAoYutKwhbTEIUlQls82unRW6kySfXqHXX5RtQpipgB8MW6PfOq3NHo5MgrOZU66iOHF5tzQpbyGDcOBh+Z25jmOtOgFesUzvbc1c3voTgpoi+EwWx5XjZhSFAr7qej4VEo3IXzQvG4Ih6BTVScOPFK25rlgme+4+0o71C0jPDs+kuVbHAVNQ7Eyjj6GKWjRko/AVGR0byjC9DLpWIUw1sjkWE0geduVXlRC3OjPfipBeCpAA6vXSQm4RTztCoRXKfJaX5JAS0vslHCMaBOYtJhDtVyb8hv62l3jOXxKDueROdEIjj5TckFkXCMLJVffUCcVUa56uZPaRuUGa020eDo0PH0odsVMB3SPeskqMhkWdeVYHAWpybBtZ5gzdHAt6QBvJ2IFILvuuYaVbY4nHkL3XDEzYZ2OJ9+ja+8SUPkXxDvpY4kDp6/orqLQXwOBOPB1RC8o2WAOMSeavQ5Uwrq1CNqf4gaOObShHJOFOtMmtQv9i+cY1YpUKU7AXYw+ovt6OX0v6jWTyTC33xB79GQfw4ow8HVDD9QGAzpbFRKbTxLRp1Pc7WVh1C/YYnm3zA3YcbB30mqVIYPjcCW5Ut2rSq3j7Eg2u9wXgVpRtcEKXA43c/i7tVjUP1nDrJG1m0MreqLCRRPo2uxeltp2bSZR73hheHekc/iKw3kghqbK6i48MS/Kb3gtFH63Z2ZVJXwaynYHQuROIva7SR+D24X/IejRdFt9cDGbc3NlSNyDM+qGtH0Wo9/rDY0i0hqHkFEY7ukPRcEcCZpGnbJQO4cbxeVMuM1CcRjf8P4K2i6W+lvdiuj09j8rsIk6lAh6OJR2w2n4VmtNxpd6++/4BWnIgFzhqhtlUnvq9JHeQi6Y09vat26zlmScUO/hnJ98S3fQogE40mzi7TSC2zwkzHSHLf3Ohyjv/n3TgvJXtJd2ouQ8r2F6mifIFz+0+8ZN+LCqTlmA2AznATTfvPfknOuUV+2Am4+iZJL/ZyoUvxz/klZQUkWBPO3yKh8/rddMX/PZVZpYeb4qjLw8pskCHNhU0WYr+RXRPj5w+N9GI+wAxSbPpugzGvW4v7r/wDPufiP"
_raw=zlib.decompress(base64.b64decode(_encoded))
assert hashlib.sha256(_raw).hexdigest()==PAYLOAD_SHA
SOURCES=json.loads(_raw.decode('utf-8'))['sources']
assert len(SOURCES)==24 and len(set(x['problem_id'] for x in SOURCES))==24
assert Counter(x['dataset'] for x in SOURCES)=={'countdown_fresh':8,'gsm8k':8,'svamp':8}
print('Frozen sources:',Counter(x['dataset'] for x in SOURCES))

## 4. Task-aware verification (no SciPy/Pandas dependency)

In [ ]:
# Shared benchmark evaluator. Countdown requires a legal expression with all numbers once.
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
PROMPT_RE=re.compile(r'Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+([-+]?(?:\d+(?:\.\d*)?|\.\d+))',re.I)
ARITH_RE=re.compile(r'[0-9eE+\-*/().=\s]{3,}')

def equals(a,b):
    return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-8)

def lastnum(text):
    x=NUM_RE.findall(str(text).replace(',',''))
    try:return float(x[-1]) if x else None
    except:return None

def eval_arith(expr):
    try:tree=pyast.parse(expr,mode='eval')
    except (SyntaxError,ValueError):return None
    numbers=[]
    def visit(node):
        if isinstance(node,pyast.Expression):return visit(node.body)
        if isinstance(node,pyast.Constant) and type(node.value) in (int,float):
            v=float(node.value)
            if not math.isfinite(v):raise ValueError('nonfinite')
            numbers.append(v);return v
        if isinstance(node,pyast.BinOp) and type(node.op) in (pyast.Add,pyast.Sub,pyast.Mult,pyast.Div):
            a,b=visit(node.left),visit(node.right)
            if isinstance(node.op,pyast.Add):return a+b
            if isinstance(node.op,pyast.Sub):return a-b
            if isinstance(node.op,pyast.Mult):return a*b
            if b==0:raise ZeroDivisionError()
            return a/b
        if isinstance(node,pyast.UnaryOp) and type(node.op) in (pyast.UAdd,pyast.USub):
            a=visit(node.operand);return -a if isinstance(node.op,pyast.USub) else a
        raise ValueError('forbidden AST')
    try:
        ans=visit(tree)
        return (ans,numbers) if math.isfinite(ans) else None
    except (ZeroDivisionError,ValueError,OverflowError):return None

def countdown_check(text,prompt):
    m=PROMPT_RE.search(prompt)
    if not m: raise ValueError('Unexpected Countdown prompt: '+prompt[:200])
    required=[float(x) for x in NUM_RE.findall(m.group(1))];goal=float(m.group(2))
    normalized=str(text).replace('×','*').replace('÷','/').replace('\\times','*')
    for chunk in ARITH_RE.findall(normalized):
        if not any(k in chunk for k in '+-*/'):continue
        for candidate in chunk.split('='):
            candidate=candidate.strip();value=eval_arith(candidate)
            if value is None:continue
            output,nums=value
            if len(nums)==len(required) and all(equals(a,b) for a,b in zip(sorted(nums),sorted(required))) and equals(output,goal):
                return True,candidate
    return False,None

def verify(text,s):
    strict_matches=FINAL_RE.findall(str(text))
    strict=float(strict_matches[-1]) if strict_matches else None
    numeric=lastnum(text)
    if s['dataset']=='countdown_fresh': success,expr=countdown_check(text,s['prompt'])
    else: success=equals(numeric,s['target']);expr=None
    return {'task_success':bool(success),'strict_success':equals(strict,s['target']),
            'numeric_success':equals(numeric,s['target']),'strict_prediction':strict,
            'numeric_prediction':numeric,'valid_expression':expr}

_demo='Use the numbers 19, 29, and 24 exactly once to produce the target 14.'
assert countdown_check('19 + 24 - 29 = 14',_demo)[0]
assert not countdown_check('29 - 19 + 24 = 14',_demo)[0]
assert not countdown_check('29-19=10 FINAL_ANSWER:14',_demo)[0]
print('Verification regression tests PASS')

## 5. Model-independent experiment engine and checkpoint format

In [ ]:
PROTOCOL='rsx1819-combined-v6-20261019'
MODELS={
    'MDLM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-mdlm-v0.1',
        sha='a284e895a6248256baf2f60502e54aba61b24c1a',
        actions=['tail25','tail50','middle25','global'],
        sampler_cls=MDLMSampler,config_cls=MDLMSamplerConfig,
    ),
    'BD3LM': dict(
        repo='dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-bd3lm-v0.1',
        sha='4bcf3a00ba6ec28d9cf9c3ebc4a5a5dc01c6023c',
        actions=['tail25','tail50','tail75','global'],
        sampler_cls=BD3LMSampler,config_cls=BD3LMSamplerConfig,
    ),
}
OUT_LEN=64; BLOCK=32; STEPS=64; TEMPERATURE=.15
GEN_SEED=38191; REPAIR_SEEDS=[38192,38193]; SELECT_MAX=8
PROMPT_SUFFIX='\nReturn a concise answer. For word problems finish with FINAL_ANSWER: <number>. For Countdown give a valid arithmetic expression using each supplied number once.'
CTX=None
SRC_MAP={s['problem_id']:s for s in SOURCES}
assert len(SRC_MAP)==24

def normalized_json(x):
    if isinstance(x,Path):return str(x)
    if isinstance(x,torch.Tensor):return x.detach().cpu().tolist()
    if hasattr(x,'item'):
        try:return x.item()
        except Exception:pass
    raise TypeError('Nonserializable '+repr(type(x)))

def atomic_json(path,value):
    path=Path(path);p=path.with_suffix(path.suffix+'.tmp')
    with p.open('w',encoding='utf-8') as f:
        json.dump(value,f,indent=2,ensure_ascii=False,default=normalized_json,allow_nan=False)
        f.flush();os.fsync(f.fileno())
    os.replace(p,path)

def append_record(path,row):
    with open(path,'a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False,default=normalized_json,allow_nan=False)+'\n')
        f.flush();os.fsync(f.fileno())

def completed_map(path,signature):
    records={}
    if not Path(path).exists():return records
    with open(path,encoding='utf-8') as f:
        for lineno,line in enumerate(f,1):
            if not line.strip():continue
            try:r=json.loads(line)
            except Exception as ex:raise ValueError(f'Malformed checkpoint JSONL {path}:{lineno}; backup before editing') from ex
            if r['signature']!=signature:raise ValueError('Checkpoint run-signature mismatch: '+str(path))
            if r['id'] in records:raise ValueError('Duplicate trial ID in checkpoint: '+str(r['id']))
            records[r['id']]=r
    return records

def model_dir(family):
    f=DRIVE/family
    f.mkdir(parents=True,exist_ok=True)
    return f

def config_signature(family):
    m=MODELS[family]
    frozen=dict(protocol=PROTOCOL,family=family,model_id=m['repo'],model_sha=m['sha'],
                source_commit=SOURCE_COMMIT,source_payload_sha=PAYLOAD_SHA,
                n_sources=len(SOURCES),out_tokens=OUT_LEN,block=BLOCK,steps=STEPS,
                temperature=TEMPERATURE,gen_seed=GEN_SEED,repair_seeds=REPAIR_SEEDS,
                select_max=SELECT_MAX,actions=m['actions'])
    return hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest(),frozen

def files_for(family):
    folder=model_dir(family)
    return {'folder':folder,'manifest':folder/'run_manifest.json',
            'natural':folder/'natural_generations.jsonl',
            'repair':folder/'repair_trials.jsonl',
            'cohort':folder/'frozen_failure_ids.json',
            'errors':folder/'errors.jsonl'}

def init_family(family):
    paths=files_for(family);sig,cfg=config_signature(family)
    if paths['manifest'].exists():
        old=json.loads(paths['manifest'].read_text())
        if old['signature']!=sig:raise RuntimeError('Different protocol found for '+family+'; do not overwrite existing run. Rename Drive folder or restore previous notebook.')
    else:
        atomic_json(paths['manifest'],dict(signature=sig,config=cfg,
            created_utc=datetime.now(timezone.utc).isoformat(),gpu=GPU,torch_version=torch.__version__,transformers_version=transformers.__version__))
    return paths,sig

def unload_model():
    global CTX
    if CTX is not None:
        print('Unloading',CTX['family'])
        CTX['sampler']=None
        CTX['model']=None
        CTX['tokenizer']=None
        CTX=None
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()

def load_family(family):
    global CTX
    assert family in MODELS
    if CTX and CTX['family']==family:
        print(family,'already loaded');return
    unload_model()
    m=MODELS[family]; paths,sig=init_family(family)
    print('Loading',family,'public Hugging Face checkpoint (no token needed)',m['repo'])
    # HF cache-first; otherwise pinned full-snapshot download; simple retry.
    try:
        local=snapshot_download(repo_id=m['repo'],revision=m['sha'],local_files_only=True)
    except Exception:
        for trial in range(3):
            try:
                local=snapshot_download(repo_id=m['repo'],revision=m['sha'],max_workers=4)
                break
            except Exception:
                if trial==2:raise
                time.sleep(2**trial)
    local=Path(local)
    assert (local/'config.json').exists() and (local/'model.safetensors').exists()
    try:
        tokenizer=AutoTokenizer.from_pretrained(str(local),local_files_only=True,trust_remote_code=False,use_fast=True)
        if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
        if tokenizer.bos_token_id is None:tokenizer.bos_token=tokenizer.pad_token
        if tokenizer.mask_token_id is None:
            tokenizer.mask_token='<|mask|>'
        mask_id=int(tokenizer.mask_token_id)
        pad_id=int(tokenizer.pad_token_id)
        eos_id=int(tokenizer.eos_token_id)
        config=A2DQwen2Config.from_pretrained(str(local),local_files_only=True)
        if config.model_type!='a2d-qwen2':raise ValueError('Wrong checkpoint architecture: '+str(config.model_type))
        model=A2DQwen2LMHeadModel.from_pretrained(
            str(local),config=config,dtype=torch.float16,
            local_files_only=True,low_cpu_mem_usage=True,attn_implementation='sdpa',
        ).to('cuda').eval()
        nrows=int(model.get_input_embeddings().weight.shape[0])
        assert 0<=mask_id<nrows and mask_id not in [pad_id,eos_id],('Mask token missing from trained vocabulary',mask_id,nrows)
        sampler=m['sampler_cls'](model=model,tokenizer=tokenizer)
        CTX=dict(family=family,model=model,tokenizer=tokenizer,sampler=sampler,
                 mask=mask_id,eos=eos_id,pad=pad_id,paths=paths,signature=sig,
                 model_sha=m['sha'],model_id=m['repo'],classes=m)
        print('MODEL LOADED',family,'GPU allocated GB',round(torch.cuda.memory_allocated()/2**30,3),'mask id',mask_id)
    except Exception as exc:
        append_record(paths['errors'],dict(stage='model_load',signature=sig,error=repr(exc),traceback=traceback.format_exc(),timestamp=datetime.now(timezone.utc).isoformat()))
        raise

def prepare_ids(s):
    t=CTX['tokenizer']
    msg=[{'role':'system','content':'You solve quantitative reasoning tasks carefully.'},
         {'role':'user','content':s['prompt']+PROMPT_SUFFIX}]
    ids=t.apply_chat_template(msg,tokenize=True,add_generation_prompt=True,enable_thinking=False)
    if isinstance(ids,dict):ids=ids['input_ids']
    ids=[int(i) for i in ids]
    assert ids and CTX['mask'] not in ids
    return ids

def visible_answer_tokens(token_ids):
    result=[]
    for i in token_ids:
        if int(i) in [CTX['eos'],CTX['pad']]:break
        result.append(int(i))
    return result

def decode(token_ids):
    return CTX['tokenizer'].decode([int(i) for i in token_ids],skip_special_tokens=True).strip()

def sampler_config(answer_budget):
    a=dict(max_new_tokens=int(answer_budget),steps=STEPS,block_size=BLOCK,
           temperature=TEMPERATURE,remasking='low_confidence',return_dict=True,
           right_shift_logits=False)
    if CTX['family']=='BD3LM':a['steps_per_block']=32
    return CTX['classes']['config_cls'](**a)

class CountCalls:
    def __init__(self,module):self.module=module;self.count=0;self.h=None
    def cb(self,*args):self.count+=1
    def __enter__(self):self.count=0;self.h=self.module.register_forward_pre_hook(self.cb);return self
    def __exit__(self,*args):self.h.remove();self.h=None

def set_seed(seed):
    torch.manual_seed(int(seed));torch.cuda.manual_seed_all(int(seed));random.seed(int(seed))

@torch.no_grad()
def real_generate(prompt,answer_budget,seed):
    set_seed(seed)
    t=time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    with CountCalls(CTX['model']) as calls:
        result=CTX['sampler'].sample([list(prompt)],sampler_config(answer_budget),return_dict=True)
    raw=result.sequences[0].detach().cpu().tolist()
    if CTX['family']=='MDLM':
        assert raw[:len(prompt)]==list(prompt), 'MDLM changed observed prompt'
        answer=raw[len(prompt):]
    else:
        offset=math.ceil(len(prompt)/BLOCK)*BLOCK
        assert raw[offset-len(prompt):offset]==list(prompt),'BD3LM changed observed prompt'
        answer=raw[offset:]
    assert not any(t==CTX['mask'] for t in answer), 'Unresolved generation mask(s)'
    assert len(answer)<=answer_budget
    return dict(token_ids=visible_answer_tokens(answer),nfe=int(calls.count),
                raw_generated_tokens=len(answer),wall_seconds=float(time.perf_counter()-t),
                peak_vram_gb=float(torch.cuda.max_memory_allocated()/2**30))

@torch.no_grad()
def real_repair(source,old_ids,action,seed):
    assert len(old_ids)>=8
    old=[int(i) for i in old_ids];L=len(old)
    if action=='global':cut=0
    elif action=='tail25':cut=max(1,math.floor(.75*L))
    elif action=='tail50':cut=max(1,math.floor(.50*L))
    elif action=='tail75':cut=max(1,math.floor(.25*L))
    elif action=='middle25':cut=None
    else:raise ValueError(action)
    set_seed(seed)
    t=time.perf_counter();torch.cuda.reset_peak_memory_stats()
    prompt=prepare_ids(source)
    if CTX['family']=='MDLM':
        visible=set(range(cut)) if cut is not None else set()
        if action=='middle25':
            width=max(1,math.ceil(.25*L));start=(L-width)//2
            visible={i for i in range(L) if not(start<=i<start+width)}
        canvas=[old[i] if i in visible else CTX['mask'] for i in range(L)]
        with CountCalls(CTX['model']) as calls:
            result=CTX['sampler'].infill([prompt+canvas],sampler_config(L),return_dict=True)
        seq=result.sequences[0].detach().cpu().tolist()
        assert seq[:len(prompt)]==prompt,'MDLM changed prompt during infill'
        new=seq[len(prompt):]
        assert len(new)==L and all(new[i]==old[i] for i in visible),'MDLM altered preserved positions'
        locked=len(visible)
        kind='native_mdlm_infill'
    else:
        assert cut is not None, 'BD3LM does not implement interior infill'
        p=prompt+old[:cut]
        with CountCalls(CTX['model']) as calls:
            result=CTX['sampler'].sample([p],sampler_config(L-cut),return_dict=True)
        seq=result.sequences[0].detach().cpu().tolist()
        offset=math.ceil(len(p)/BLOCK)*BLOCK
        assert seq[offset-len(p):offset]==p,'BD3LM changed observed prefix'
        new=old[:cut]+seq[offset:]
        assert new[:cut]==old[:cut]
        locked=cut
        kind='native_bd3lm_suffix_generation'
    assert not any(t==CTX['mask'] for t in new),'Repair left unresolved mask tokens'
    return dict(token_ids=visible_answer_tokens(new),raw_repaired_ids=[int(t) for t in new],nfe=int(calls.count),locked_tokens=locked,
                canvas_tokens=L,intervention_kind=kind,
                wall_seconds=float(time.perf_counter()-t),
                peak_vram_gb=float(torch.cuda.max_memory_allocated()/2**30))

def run_source(s):
    r=real_generate(prepare_ids(s),OUT_LEN,GEN_SEED)
    checks=verify(decode(r['token_ids']),s)
    return dict(id=s['problem_id']+'|'+str(GEN_SEED),problem_id=s['problem_id'],
        dataset=s['dataset'],prompt=s['prompt'],target=s['target'],generation_seed=GEN_SEED,
        output=decode(r['token_ids']),verification=checks,task_success=bool(checks['task_success']),**r)

def select_failure_cohort():
    p=CTX['paths'];natural=completed_map(p['natural'],CTX['signature'])
    if len(natural)!=24:raise RuntimeError('Finish all six natural-generation batch cells before selecting failures.')
    failures=[r for r in natural.values() if not r['task_success'] and len(r['token_ids'])>=8]
    selected=[]
    for ds in ['countdown_fresh','gsm8k','svamp']:
        group=sorted([r for r in failures if r['dataset']==ds],key=lambda r:hashlib.sha256((CTX['family']+'|'+r['id']).encode()).hexdigest())
        selected+=group[:2]
    pool=sorted([r for r in failures if r['id'] not in {x['id'] for x in selected}],key=lambda r:hashlib.sha256((CTX['family']+'|'+r['id']).encode()).hexdigest())
    selected=(selected+pool[:max(0,SELECT_MAX-len(selected))])[:SELECT_MAX]
    assert len({r['problem_id'] for r in selected})==len(selected)
    proposed=[r['id'] for r in selected]
    if p['cohort'].exists():
        stored=json.loads(p['cohort'].read_text())
        if stored!=proposed:raise RuntimeError('Frozen cohort differs from current selection. Investigate before continuing.')
    else:atomic_json(p['cohort'],proposed)
    print('Selected',len(selected),'failures; dataset counts',dict(Counter(x['dataset'] for x in selected)))
    return selected

def make_repair_tasks():
    selected=select_failure_cohort()
    return [(r,action,seed) for r in selected for action in CTX['classes']['actions'] for seed in REPAIR_SEEDS]

def run_repair_one(failure,action,seed):
    src=SRC_MAP[failure['problem_id']]
    r=real_repair(src,failure['token_ids'],action,seed)
    answer=decode(r['token_ids']);checks=verify(answer,src)
    orig=failure['token_ids'];fixed=r['raw_repaired_ids']
    preserved=sum(a==b for a,b in zip(orig,fixed))/max(len(orig),len(fixed),1)
    prior=failure['verification']['numeric_prediction'];after=checks['numeric_prediction']
    return dict(id=f"{failure['id']}|{action}|{seed}",failure_id=failure['id'],problem_id=src['problem_id'],
                dataset=src['dataset'],action=action,repair_seed=seed,output=answer,verification=checks,
                task_success=bool(checks['task_success']),strict_success=bool(checks['strict_success']),
                numeric_success=bool(checks['numeric_success']),
                same_wrong_numeric=bool(not checks['task_success'] and prior is not None and after is not None and equals(prior,after)),
                token_position_preservation=float(preserved),forced_fraction=float(r['locked_tokens']/len(orig)),**r)

# Safeguard: reject past checkpoint records from another experiment instead of overwriting.
for name in MODELS:
    initialize_signature=config_signature(name)[0]
    print(name,'signature',initialize_signature[:16])
print('Experiment engine ready.')

## 6. Real GPU smoke tests and short-batch runner

In [ ]:
def smoke(family):
    assert CTX and CTX['family']==family
    src=SOURCES[0]
    first=real_generate(prepare_ids(src),16,38400)
    print(family,'smoke generation: NFE',first['nfe'],'preview',decode(first['token_ids'])[:140])
    ids=first['token_ids']
    if len(ids)<8:
        ids=CTX['tokenizer']('FINAL_ANSWER: 0. This answer is incorrect.',add_special_tokens=False)['input_ids']
        ids=[i for i in ids if i not in (CTX['eos'],CTX['pad'])]
        assert len(ids)>=8
    second=real_repair(src,ids,'global',38401)
    assert first['nfe']>=1 and second['nfe']>=1
    print(family,'smoke repair: NFE',second['nfe'],'preservation',second['locked_tokens'])
    print('SUCCESS: Real CUDA forward passes observed.')

def run_batch(stage,start,count=4,max_minutes=8):
    assert CTX and stage in ('natural','repair')
    path=CTX['paths'][stage];sig=CTX['signature']
    existing=completed_map(path,sig)
    if stage=='natural':
        tasks=SOURCES
    else:
        tasks=make_repair_tasks()
    end=min(start+count,len(tasks))
    began=time.monotonic();new=0;skipped=0
    for index in range(start,end):
        if stage=='natural':
            s=tasks[index];uid=f"{s['problem_id']}|{GEN_SEED}"
        else:
            failure,action,seed=tasks[index]
            uid=f"{failure['id']}|{action}|{seed}"
        if uid in existing:
            skipped+=1;continue
        if new>0 and time.monotonic()-began >= max_minutes*60:
            print('Soft cell time limit reached; run this same batch cell again after a break to finish remaining tasks.')
            break
        try:
            row=run_source(s) if stage=='natural' else run_repair_one(failure,action,seed)
        except Exception as e:
            append_record(CTX['paths']['errors'],dict(signature=sig,stage=stage,id=uid,error=repr(e),traceback=traceback.format_exc(),utc=datetime.now(timezone.utc).isoformat()))
            print('Error checkpointed in Drive:',CTX['paths']['errors'])
            raise
        row.update(signature=sig,status='ok')
        append_record(path,row)
        existing[uid]=row
        new+=1
        print(f'{CTX["family"]} {stage}: {index+1}/{len(tasks)} | success={row["task_success"]} | real NFE={row["nfe"]} | seconds={row["wall_seconds"]:.1f}')
    remaining=[]
    for idx in range(start,end):
        if stage=='natural': uid=f"{tasks[idx]['problem_id']}|{GEN_SEED}"
        else:
            fail,action,seed=tasks[idx];uid=f"{fail['id']}|{action}|{seed}"
        if uid not in existing: remaining.append(idx)
    print(f'Batch {stage} [{start}:{end}]: {new} newly completed; {skipped} already done; {len(remaining)} still pending in this batch.')
    if remaining:print('RERUN THIS CELL to finish the batch. Google Drive records remain intact.')
    return dict(new=new,skipped=skipped,pending=remaining)

def progress(family):
    paths,sig=init_family(family)
    naturals=completed_map(paths['natural'],sig)
    repairs=completed_map(paths['repair'],sig)
    print(family,'natural generations',len(naturals),'/24','completed repair trials',len(repairs))
    if paths['cohort'].exists():
        n=len(json.loads(paths['cohort'].read_text()))
        print('Frozen failures:',n,'expected repairs:',n*4*len(REPAIR_SEEDS))

# Phase A — MDLM

Load the MDLM model and run its two real GPU smoke tests. Then execute **six generation batches** and **up to sixteen four-trial repair batches**. Each successful trial is saved to Google Drive instantly.

To resume after a disconnect: run cells 0–6 again, load MDLM here, then rerun the batch cells. Previously finished trials are skipped. Never delete `MyDrive/RepairScopeX/RSX18_19_Combined_v6`.

In [ ]:
load_family('MDLM')
smoke('MDLM')
progress('MDLM')

### Phase A1. MDLM natural generation — six independent batches

In [ ]:
run_batch('natural',start=0,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=4,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=8,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=12,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=16,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=20,count=4,max_minutes=8)

### Phase A2. Freeze MDLM failures

In [ ]:
select_failure_cohort()
progress('MDLM')

### Phase A3. MDLM repair — sixteen independent batches (four trials each)

In [ ]:
run_batch('repair',start=0,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=4,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=8,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=12,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=16,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=20,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=24,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=28,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=32,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=36,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=40,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=44,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=48,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=52,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=56,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=60,count=4,max_minutes=8)

## 7. Model-level analysis + export helpers

In [ ]:
def percentile(vals,q):
    a=sorted(vals)
    if not a:return float('nan')
    j=(len(a)-1)*q;lo=int(math.floor(j));hi=int(math.ceil(j))
    return a[lo]*(hi-j)+a[hi]*(j-lo) if lo!=hi else a[lo]

def exact_sign_test(wins,losses):
    n=wins+losses
    if not n:return 1.0
    p=min(1.0,2*sum(math.comb(n,j) for j in range(0,min(wins,losses)+1))/(2**n))
    return float(p)

def csv_write(path,rows):
    keys=sorted({k for r in rows for k in r}) if rows else ['empty']
    with open(path,'w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=keys);w.writeheader()
        for row in rows:w.writerow(row)

def export_family(family):
    paths,sig=init_family(family)
    naturals=list(completed_map(paths['natural'],sig).values())
    trials=list(completed_map(paths['repair'],sig).values())
    assert len(naturals)==24,'Incomplete 24 natural generations; rerun pending generation batch cells.'
    cohort=json.loads(paths['cohort'].read_text()) if paths['cohort'].exists() else []
    expected=len(cohort)*len(MODELS[family]['actions'])*len(REPAIR_SEEDS)
    assert len(trials)==expected,(len(trials),expected,'Rerun repair batch cells if incomplete.')
    assert len({t['id'] for t in trials})==len(trials)
    action_summary=[];dataset_summary=[];ci_summary=[];pair_summary=[]
    for action in MODELS[family]['actions']:
        selected=[t for t in trials if t['action']==action]
        if not selected:continue
        wrong=[t for t in selected if not t['task_success']]
        action_summary.append(dict(action=action,n=len(selected),successes=sum(t['task_success'] for t in selected),
                success_rate=sum(t['task_success'] for t in selected)/len(selected),
                mean_nfe=sum(t['nfe'] for t in selected)/len(selected),
                mean_wall_seconds=sum(t['wall_seconds'] for t in selected)/len(selected),
                mean_preservation=sum(t['token_position_preservation'] for t in selected)/len(selected),
                recurrence_conditional_failures=(sum(t['same_wrong_numeric'] for t in wrong)/len(wrong) if wrong else 0.0)))
        by_problem={}
        for t in selected:by_problem.setdefault(t['problem_id'],[]).append(int(t['task_success']))
        per=[sum(v)/len(v) for v in by_problem.values()]
        rng=random.Random(7800+MODELS[family]['actions'].index(action))
        boot=[sum(rng.choices(per,k=len(per)))/len(per) for _ in range(4000)]
        ci_summary.append(dict(action=action,problem_count=len(per),rate=sum(per)/len(per),
                               ci_low=percentile(boot,.025),ci_high=percentile(boot,.975)))
    for ds in ['countdown_fresh','gsm8k','svamp']:
        for action in MODELS[family]['actions']:
            subset=[t for t in trials if t['dataset']==ds and t['action']==action]
            if subset:dataset_summary.append(dict(dataset=ds,action=action,n=len(subset),success_rate=sum(t['task_success'] for t in subset)/len(subset)))
    byaction={}
    for t in trials:byaction.setdefault((t['problem_id'],t['action']),[]).append(int(t['task_success']))
    for action in MODELS[family]['actions']:
        if action=='global':continue
        diffs=[]
        for problem in {t['problem_id'] for t in trials}:
            a=byaction.get((problem,action),[]);g=byaction.get((problem,'global'),[])
            if a and g:diffs.append(sum(a)/len(a)-sum(g)/len(g))
        wins=sum(x>0 for x in diffs);losses=sum(x<0 for x in diffs)
        pair_summary.append(dict(action=action,problem_count=len(diffs),difference=sum(diffs)/len(diffs) if diffs else None,
                                 wins=wins,losses=losses,p_two_sided=exact_sign_test(wins,losses)))
    per_fail={}
    for t in trials:per_fail.setdefault(t['problem_id'],{}).setdefault(t['action'],[]).append(t['task_success'])
    oracle=(sum(max(sum(s)/len(s) for s in acts.values()) for acts in per_fail.values())/len(per_fail)) if per_fail else 0.
    best=max((r['success_rate'] for r in action_summary),default=0)
    gate=dict(family=family,model=MODELS[family]['repo'],model_sha=MODELS[family]['sha'],
              natural_generations=len(naturals),natural_failures=sum(not x['task_success'] for x in naturals),
              selected_failures=len(cohort),repair_trials=len(trials),repair_successes=sum(t['task_success'] for t in trials),
              native_intervention='infill' if family=='MDLM' else 'prefix-conditioned_suffix',
              all_preservation_audits=all(t['token_position_preservation']+1e-9>=t['forced_fraction'] for t in trials),
              in_sample_oracle=oracle,best_fixed_success=best,exploratory_oracle_headroom=oracle-best,
              complete=True,successful_repair_signal=sum(t['task_success'] for t in trials)>=4,
              statistical_caveat='Exploratory pilot; problem-cluster sampling; no pooled NFE comparisons between architectures; heuristic last-number verifier requires audit.')
    atomic_json(paths['folder']/'GATE.json',gate)
    for filename,rows in [('action_summary.csv',action_summary),('dataset_summary.csv',dataset_summary),
                          ('cluster_bootstrap.csv',ci_summary),('problem_sign_tests.csv',pair_summary)]:
        csv_write(paths['folder']/filename,rows)
    manifest={}
    for p in sorted(paths['folder'].iterdir()):
        if p.is_file() and p.name!='artifact_manifest.json':manifest[p.name]=hashlib.sha256(p.read_bytes()).hexdigest()
    atomic_json(paths['folder']/'artifact_manifest.json',dict(signature=sig,files=manifest))
    zippath=Path('/content')/f'RSX_{family}_Combined_v6_results.zip'
    with zipfile.ZipFile(zippath,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(paths['folder'].iterdir()):
            if p.is_file():z.write(p,arcname=p.name)
    print(family,'FINAL',json.dumps(gate,indent=2))
    print('ZIP',zippath,'bytes',zippath.stat().st_size)
    return str(zippath)

### Phase A4. Export MDLM results and free GPU memory

In [ ]:
MDLM_ZIP=export_family('MDLM')
from google.colab import files
files.download(MDLM_ZIP)
unload_model()

# Phase B — BD3LM

The previous cell unloads MDLM before BD3LM is loaded. This is necessary because both 0.5B checkpoints should not remain on GPU concurrently. **Do not run this phase if MDLM is still loaded without unloading first**.

For resuming only BD3LM: run setup and shared helper cells at the top, **skip the MDLM load/sweep**, then run the next cell. The checkpoint manifest is per model.

In [ ]:
load_family('BD3LM')
smoke('BD3LM')
progress('BD3LM')

### Phase B1. BD3LM natural generation — six independent batches

In [ ]:
run_batch('natural',start=0,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=4,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=8,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=12,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=16,count=4,max_minutes=8)

In [ ]:
run_batch('natural',start=20,count=4,max_minutes=8)

### Phase B2. Freeze BD3LM failures

In [ ]:
select_failure_cohort()
progress('BD3LM')

### Phase B3. BD3LM repair — sixteen independent batches (four trials each)

In [ ]:
run_batch('repair',start=0,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=4,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=8,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=12,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=16,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=20,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=24,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=28,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=32,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=36,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=40,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=44,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=48,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=52,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=56,count=4,max_minutes=8)

In [ ]:
run_batch('repair',start=60,count=4,max_minutes=8)

### Phase B4. Export BD3LM results and free GPU memory

In [ ]:
BD3LM_ZIP=export_family('BD3LM')
from google.colab import files
files.download(BD3LM_ZIP)
unload_model()

## 8. Combined ZIP and recovery diagnostics

In [ ]:
# Does not require models to be loaded. Export finished and partial files in one archive.
combined=Path('/content/RepairScopeX_RSX18_RSX19_Combined_v6_results.zip')
with zipfile.ZipFile(combined,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for fam in MODELS:
        folder=model_dir(fam)
        for p in sorted(folder.iterdir()):
            if p.is_file():z.write(p,arcname=fam+'/'+p.name)
print('Combined result/diagnostic ZIP:',combined,'size MB',round(combined.stat().st_size/1024**2,3))
from google.colab import files
files.download(str(combined))
for fam in MODELS:progress(fam)

### Protocol and interpretation cautions

The two models share Qwen-based backbone training, so they are not completely independent models. Their repair interfaces differ: MDLM can reopen interior spans; BD3LM's released native sampler only supports causal blocks conditioned on a frozen prefix. **Do not pool the two repair methods as identical interventions.**

Numeric-last-answer extraction on GSM8K/SVAMP remains a provisional heuristic. Countdown correctness requires a full arithmetic-expression check. Review positive claims against raw generations. Failed or interrupted tasks are not represented as completed trials and are logged in `errors.jsonl`.

Report the **two generated ZIP files** `RSX_MDLM_Combined_v6_results.zip` and `RSX_BD3LM_Combined_v6_results.zip` (or the combined ZIP). Real runs may have no successful repairs; in that case the notebook reports the negative result rather than manufacturing a positive signal.